# Fase 2: Compresión SVD de BERT fine-tuned

Aplicamos descomposición SVD (Singular Value Decomposition) a las capas lineales del
modelo BERT fine-tuned en GoEmotions para analizar el trade-off calidad/compresión.

In [ ]:
# Instalación de dependencias (para Google Colab)
# !pip install transformers datasets torch scikit-learn accelerate seaborn

In [ ]:
import sys
import os

# En Colab, montar Drive y ajustar path al proyecto
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_ROOT = '/content/drive/MyDrive/transformer-structural-compression-v2'

# En local
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

from src.data import load_goemotions
from src.data.dataset import NUM_LABELS, EMOTION_NAMES
from src.utils import compute_metrics
from src.compression import (
    apply_svd_compression,
    count_parameters,
    get_compression_ratio,
    compute_singular_value_energy,
    get_target_layer_names,
)

sns.set_style("whitegrid")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## 1. Cargar modelo fine-tuned y datos de test

In [ ]:
model_path = os.path.join(PROJECT_ROOT, "results", "bert-goemotions-final")

baseline_model = AutoModelForSequenceClassification.from_pretrained(
    model_path,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
)
baseline_model.eval()

params = count_parameters(baseline_model)
print(f"Parámetros totales: {params['total']:,}")
print(f"Parámetros entrenables: {params['trainable']:,}")

In [ ]:
_, _, test_ds, emotion_names, data_collator = load_goemotions()
print(f"Test set: {len(test_ds)} ejemplos")

In [ ]:
# Verificar que el baseline reproduce las métricas de Fase 1
eval_args = TrainingArguments(
    output_dir=os.path.join(PROJECT_ROOT, "results", "tmp_eval"),
    per_device_eval_batch_size=64,
    fp16=(device == "cuda"),
    report_to="none",
)

trainer = Trainer(
    model=baseline_model,
    args=eval_args,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

baseline_metrics = trainer.evaluate(test_ds)
print(f"Baseline F1 macro: {baseline_metrics['eval_f1_macro']:.4f}")
print(f"Baseline F1 micro: {baseline_metrics['eval_f1_micro']:.4f}")

## 2. Análisis del espectro de valores singulares

In [ ]:
energy_info = compute_singular_value_energy(baseline_model)

# Tabla resumen de rangos necesarios para distintos umbrales de energía
rows = []
for name, info in energy_info.items():
    # Extraer componente corto: e.g. "layer.0.attention.self.query"
    short_name = name.replace("bert.encoder.", "")
    rows.append({
        "layer": short_name,
        "rank_90%": info["rank_for_90"],
        "rank_95%": info["rank_for_95"],
        "rank_99%": info["rank_for_99"],
    })

energy_df = pd.DataFrame(rows)
print(f"Total capas target: {len(energy_df)}")
print(f"\nRango medio para 90% energía: {energy_df['rank_90%'].mean():.0f}")
print(f"Rango medio para 95% energía: {energy_df['rank_95%'].mean():.0f}")
print(f"Rango medio para 99% energía: {energy_df['rank_99%'].mean():.0f}")
energy_df

In [ ]:
# Heatmap: rango para 90% de energía por bloque y componente
component_types = ["attention.self.query", "attention.self.key", "attention.self.value",
                   "attention.output.dense", "intermediate.dense", "output.dense"]
component_short = ["Q", "K", "V", "Attn Out", "FFN Up", "FFN Down"]
num_layers = 12

heatmap_data = np.zeros((num_layers, len(component_types)))
for name, info in energy_info.items():
    for j, comp in enumerate(component_types):
        if comp in name:
            layer_idx = int(name.split(".")[3])  # bert.encoder.layer.X
            heatmap_data[layer_idx, j] = info["rank_for_90"]
            break

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(heatmap_data, annot=True, fmt=".0f", cmap="YlOrRd",
            xticklabels=component_short,
            yticklabels=[f"Layer {i}" for i in range(num_layers)],
            ax=ax)
ax.set_title("Rango necesario para 90% de energía por capa/componente")
ax.set_xlabel("Componente")
ax.set_ylabel("Capa del encoder")
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "results", "svd_energy_heatmap.png"), dpi=150)
plt.show()

In [ ]:
# Curvas de decaimiento de valores singulares (una línea por tipo de componente)
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for j, (comp, short) in enumerate(zip(component_types, component_short)):
    ax = axes[j]
    for name, info in energy_info.items():
        if comp in name:
            layer_idx = int(name.split(".")[3])
            sv = info["singular_values"].numpy()
            ax.plot(sv / sv[0], alpha=0.6, label=f"L{layer_idx}")
    ax.set_title(short)
    ax.set_xlabel("Índice")
    ax.set_ylabel("Valor singular (normalizado)")
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=6, ncol=2)

fig.suptitle("Decaimiento de valores singulares por tipo de componente", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "results", "svd_decay_curves.png"), dpi=150)
plt.show()

## 3. Curva analítica de parámetros vs rango

In [ ]:
# Calcular reducción teórica de parámetros sin instanciar modelos
target_names = get_target_layer_names(baseline_model)

# Recoger dimensiones de las capas target
layer_dims = []
original_params_in_targets = 0
for name in target_names:
    parts = name.split(".")
    module = baseline_model
    for p in parts:
        module = getattr(module, p)
    m, n = module.weight.shape  # (out_features, in_features)
    bias_params = m if module.bias is not None else 0
    layer_dims.append((m, n, bias_params))
    original_params_in_targets += m * n + bias_params

non_target_params = count_parameters(baseline_model)["total"] - original_params_in_targets

ranks = list(range(1, 769, 1))
total_params_by_rank = []
for rank in ranks:
    compressed_in_targets = 0
    for m, n, bias_params in layer_dims:
        k = min(rank, min(m, n))  # rank can't exceed matrix dimensions
        compressed_in_targets += n * k + m * k + bias_params  # first(n,k) + second(k,m) + bias
    total_params_by_rank.append(non_target_params + compressed_in_targets)

original_total = count_parameters(baseline_model)["total"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(ranks, [p / 1e6 for p in total_params_by_rank])
ax1.axhline(y=original_total / 1e6, color="red", linestyle="--", label="Original")
ax1.set_xlabel("Rango SVD")
ax1.set_ylabel("Parámetros totales (M)")
ax1.set_title("Parámetros totales vs rango SVD")
ax1.legend()

ax2.plot(ranks, [p / original_total * 100 for p in total_params_by_rank])
ax2.axhline(y=100, color="red", linestyle="--", label="100% (original)")
for r in [64, 128, 256, 384, 512]:
    idx = r - 1
    pct = total_params_by_rank[idx] / original_total * 100
    ax2.plot(r, pct, "o", markersize=6)
    ax2.annotate(f"r={r}\n{pct:.1f}%", (r, pct), textcoords="offset points",
                 xytext=(5, 10), fontsize=8)
ax2.set_xlabel("Rango SVD")
ax2.set_ylabel("% de parámetros originales")
ax2.set_title("Retención de parámetros vs rango SVD")
ax2.legend()

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "results", "svd_params_vs_rank.png"), dpi=150)
plt.show()

## 4. Experimento principal: SVD a distintos rangos

In [ ]:
svd_ranks = [64, 128, 256, 384, 512]
results = []

for rank in svd_ranks:
    print(f"\n{'='*60}")
    print(f"SVD rank = {rank}")
    print(f"{'='*60}")

    compressed_model = apply_svd_compression(baseline_model, rank=rank)
    compressed_model.to(device)

    comp_params = count_parameters(compressed_model)
    ratio = get_compression_ratio(baseline_model, compressed_model)
    print(f"Parámetros: {comp_params['total']:,} (ratio: {ratio:.2f}x)")

    eval_trainer = Trainer(
        model=compressed_model,
        args=eval_args,
        compute_metrics=compute_metrics,
        data_collator=data_collator,
    )
    metrics = eval_trainer.evaluate(test_ds)

    f1_macro = metrics["eval_f1_macro"]
    f1_micro = metrics["eval_f1_micro"]
    print(f"F1 macro: {f1_macro:.4f}  |  F1 micro: {f1_micro:.4f}")

    row = {
        "rank": rank,
        "params": comp_params["total"],
        "compression_ratio": ratio,
        "f1_macro": f1_macro,
        "f1_micro": f1_micro,
    }
    # F1 por emoción
    for i, name in enumerate(emotion_names):
        row[f"f1_{name}"] = metrics[f"eval_f1_label_{i}"]
    results.append(row)

    # Liberar memoria
    del compressed_model, eval_trainer
    if device == "cuda":
        torch.cuda.empty_cache()

results_df = pd.DataFrame(results)
results_df

In [ ]:
# Agregar baseline como referencia
baseline_f1_macro = baseline_metrics["eval_f1_macro"]
baseline_f1_micro = baseline_metrics["eval_f1_micro"]

results_df["f1_macro_retention"] = results_df["f1_macro"] / baseline_f1_macro * 100
results_df["f1_micro_retention"] = results_df["f1_micro"] / baseline_f1_micro * 100
results_df["param_reduction_pct"] = (1 - results_df["params"] / count_parameters(baseline_model)["total"]) * 100

print(results_df[["rank", "params", "compression_ratio", "f1_macro", "f1_micro",
                   "f1_macro_retention", "param_reduction_pct"]].to_string(index=False))

## 5. Visualizaciones

In [ ]:
# 5a. F1 macro/micro vs rango SVD
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(results_df["rank"], results_df["f1_macro"], "o-", label="F1 macro", linewidth=2)
ax.plot(results_df["rank"], results_df["f1_micro"], "s-", label="F1 micro", linewidth=2)
ax.axhline(y=baseline_f1_macro, color="blue", linestyle="--", alpha=0.5, label=f"Baseline macro ({baseline_f1_macro:.4f})")
ax.axhline(y=baseline_f1_micro, color="orange", linestyle="--", alpha=0.5, label=f"Baseline micro ({baseline_f1_micro:.4f})")

ax.set_xlabel("Rango SVD")
ax.set_ylabel("F1 Score")
ax.set_title("F1 Score vs Rango SVD")
ax.legend()
ax.set_xticks(svd_ranks)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "results", "svd_f1_vs_rank.png"), dpi=150)
plt.show()

In [ ]:
# 5b. Retención de F1 vs ratio de compresión
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(results_df["compression_ratio"], results_df["f1_macro_retention"], "o-", label="F1 macro", linewidth=2)
ax.plot(results_df["compression_ratio"], results_df["f1_micro_retention"], "s-", label="F1 micro", linewidth=2)

for _, row in results_df.iterrows():
    ax.annotate(f"r={int(row['rank'])}",
                (row["compression_ratio"], row["f1_macro_retention"]),
                textcoords="offset points", xytext=(5, 10), fontsize=9)

ax.axhline(y=100, color="gray", linestyle="--", alpha=0.5, label="100% (baseline)")
ax.set_xlabel("Ratio de compresión (x)")
ax.set_ylabel("Retención de F1 (%)")
ax.set_title("Retención de F1 vs Ratio de compresión")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "results", "svd_retention_vs_compression.png"), dpi=150)
plt.show()

In [ ]:
# 5c. Heatmap de F1 por emoción a distintos rangos
emotion_cols = [f"f1_{name}" for name in emotion_names]
heatmap_df = results_df.set_index("rank")[emotion_cols]
heatmap_df.columns = emotion_names

# Agregar baseline
baseline_row = {name: baseline_metrics[f"eval_f1_label_{i}"] for i, name in enumerate(emotion_names)}
baseline_series = pd.DataFrame([baseline_row], index=["baseline"])
baseline_series.columns = emotion_names
heatmap_df = pd.concat([baseline_series, heatmap_df])

fig, ax = plt.subplots(figsize=(18, 5))
sns.heatmap(heatmap_df, annot=True, fmt=".2f", cmap="RdYlGn", vmin=0, vmax=1,
            ax=ax, linewidths=0.5)
ax.set_title("F1 por emoción a distintos rangos SVD")
ax.set_xlabel("Emoción")
ax.set_ylabel("Rango SVD")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "results", "svd_f1_per_emotion_heatmap.png"), dpi=150)
plt.show()

In [ ]:
# 5d. Caída de F1 por emoción a rango 128 (o el rango con mayor compresión razonable)
analysis_rank = 128
rank_row = results_df[results_df["rank"] == analysis_rank].iloc[0]

f1_baseline_per_emotion = [baseline_metrics[f"eval_f1_label_{i}"] for i in range(len(emotion_names))]
f1_compressed_per_emotion = [rank_row[f"f1_{name}"] for name in emotion_names]
f1_drop = [b - c for b, c in zip(f1_baseline_per_emotion, f1_compressed_per_emotion)]

drop_df = pd.DataFrame({
    "emotion": emotion_names,
    "baseline": f1_baseline_per_emotion,
    "compressed": f1_compressed_per_emotion,
    "drop": f1_drop,
}).sort_values("drop", ascending=False)

fig, ax = plt.subplots(figsize=(14, 6))
colors = ["red" if d > 0.05 else "orange" if d > 0.02 else "green" for d in drop_df["drop"]]
ax.barh(drop_df["emotion"], drop_df["drop"], color=colors)
ax.set_xlabel("Caída de F1")
ax.set_title(f"Caída de F1 por emoción (baseline → SVD rank={analysis_rank})")
ax.axvline(x=0, color="black", linewidth=0.5)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "results", "svd_f1_drop_per_emotion.png"), dpi=150)
plt.show()

## 6. Análisis por componente: Atención vs FFN

In [ ]:
# Comprimir solo capas de atención vs solo FFN para ver cuál tolera más compresión
all_target_names = get_target_layer_names(baseline_model)
attention_names = [n for n in all_target_names if "attention" in n]
ffn_names = [n for n in all_target_names if "intermediate" in n or "output.dense" in n
             and "attention" not in n]

print(f"Capas de atención: {len(attention_names)}")
print(f"Capas FFN: {len(ffn_names)}")

component_results = []
test_rank = 128

for component, names, label in [
    ("attention", attention_names, "Solo atención"),
    ("ffn", ffn_names, "Solo FFN"),
    ("all", None, "Todas las capas"),
]:
    print(f"\nComprimiendo: {label} (rank={test_rank})")
    compressed = apply_svd_compression(baseline_model, rank=test_rank, layer_names=names)
    compressed.to(device)

    eval_trainer = Trainer(
        model=compressed,
        args=eval_args,
        compute_metrics=compute_metrics,
        data_collator=data_collator,
    )
    metrics = eval_trainer.evaluate(test_ds)

    component_results.append({
        "component": label,
        "params": count_parameters(compressed)["total"],
        "f1_macro": metrics["eval_f1_macro"],
        "f1_micro": metrics["eval_f1_micro"],
    })
    print(f"  F1 macro: {metrics['eval_f1_macro']:.4f}  |  F1 micro: {metrics['eval_f1_micro']:.4f}")

    del compressed, eval_trainer
    if device == "cuda":
        torch.cuda.empty_cache()

component_df = pd.DataFrame(component_results)
component_df

## 7. Fine-tuning post-SVD (opcional)

In [ ]:
# Cargar datos de entrenamiento para fine-tuning post-SVD
train_ds, val_ds, _, _, _ = load_goemotions()

finetune_rank = 128  # Ajustar al rango de interés
ft_model = apply_svd_compression(baseline_model, rank=finetune_rank)
ft_model.to(device)

# Evaluar antes del fine-tuning
pre_ft_trainer = Trainer(
    model=ft_model,
    args=eval_args,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)
pre_ft_metrics = pre_ft_trainer.evaluate(test_ds)
print(f"Pre fine-tuning  — F1 macro: {pre_ft_metrics['eval_f1_macro']:.4f}")

In [ ]:
ft_args = TrainingArguments(
    output_dir=os.path.join(PROJECT_ROOT, "results", f"svd-ft-rank{finetune_rank}"),
    num_train_epochs=2,
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=100,
    fp16=(device == "cuda"),
    report_to="none",
    seed=42,
)

ft_trainer = Trainer(
    model=ft_model,
    args=ft_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

ft_trainer.train()

In [ ]:
# Evaluar después del fine-tuning
post_ft_metrics = ft_trainer.evaluate(test_ds)
print(f"Post fine-tuning — F1 macro: {post_ft_metrics['eval_f1_macro']:.4f}")
print(f"\nRecuperación:")
print(f"  Pre-FT:  {pre_ft_metrics['eval_f1_macro']:.4f}")
print(f"  Post-FT: {post_ft_metrics['eval_f1_macro']:.4f}")
print(f"  Baseline: {baseline_f1_macro:.4f}")
recovery = (post_ft_metrics['eval_f1_macro'] - pre_ft_metrics['eval_f1_macro']) / (
    baseline_f1_macro - pre_ft_metrics['eval_f1_macro']) * 100 if baseline_f1_macro != pre_ft_metrics['eval_f1_macro'] else 0
print(f"  Recuperación: {recovery:.1f}% del gap")

## 8. Guardar modelos y métricas

In [ ]:
# Guardar resultados del experimento SVD
results_path = os.path.join(PROJECT_ROOT, "results", "svd_results.csv")
results_df.to_csv(results_path, index=False)
print(f"Resultados guardados en: {results_path}")

# Guardar métricas completas en JSON
svd_metrics = {
    "baseline": {
        "f1_macro": baseline_f1_macro,
        "f1_micro": baseline_f1_micro,
        "params": count_parameters(baseline_model)["total"],
    },
    "svd_results": results_df.to_dict(orient="records"),
    "component_analysis": component_df.to_dict(orient="records") if "component_df" in dir() else None,
    "post_svd_finetuning": {
        "rank": finetune_rank,
        "pre_ft_f1_macro": pre_ft_metrics["eval_f1_macro"],
        "post_ft_f1_macro": post_ft_metrics["eval_f1_macro"],
    } if "post_ft_metrics" in dir() else None,
}

metrics_json_path = os.path.join(PROJECT_ROOT, "results", "svd_metrics.json")
with open(metrics_json_path, "w") as f:
    json.dump(svd_metrics, f, indent=2, default=float)
print(f"Métricas JSON guardadas en: {metrics_json_path}")

In [ ]:
# Guardar el mejor modelo comprimido (con y sin fine-tuning)
# Nota: usamos torch.save porque SVDLinear es un módulo custom
best_rank = results_df.loc[results_df["f1_macro_retention"].idxmax(), "rank"]
best_rank = int(best_rank)
print(f"Mejor rango (mayor retención): {best_rank}")

# Guardar modelo comprimido sin fine-tuning
best_compressed = apply_svd_compression(baseline_model, rank=best_rank)
save_path = os.path.join(PROJECT_ROOT, "results", f"bert-svd-rank{best_rank}")
os.makedirs(save_path, exist_ok=True)
torch.save({
    "model_state_dict": best_compressed.state_dict(),
    "rank": best_rank,
    "config": best_compressed.config.to_dict(),
    "compression_ratio": get_compression_ratio(baseline_model, best_compressed),
}, os.path.join(save_path, "model.pt"))
print(f"Modelo comprimido guardado en: {save_path}")

# Guardar modelo post fine-tuning si existe
if "ft_model" in dir():
    ft_save_path = os.path.join(PROJECT_ROOT, "results", f"bert-svd-rank{finetune_rank}-ft")
    os.makedirs(ft_save_path, exist_ok=True)
    torch.save({
        "model_state_dict": ft_model.state_dict(),
        "rank": finetune_rank,
        "config": ft_model.config.to_dict(),
    }, os.path.join(ft_save_path, "model.pt"))
    print(f"Modelo post-FT guardado en: {ft_save_path}")

In [ ]:
# Resumen final
print("="*60)
print("RESUMEN — Fase 2: Compresión SVD")
print("="*60)
print(f"\nBaseline: F1 macro = {baseline_f1_macro:.4f}, F1 micro = {baseline_f1_micro:.4f}")
print(f"Parámetros baseline: {count_parameters(baseline_model)['total']:,}")
print(f"\nResultados por rango:")
for _, row in results_df.iterrows():
    print(f"  rank={int(row['rank']):>3d}  |  params={int(row['params']):>12,}  |  "
          f"ratio={row['compression_ratio']:.2f}x  |  "
          f"F1 macro={row['f1_macro']:.4f} ({row['f1_macro_retention']:.1f}%)  |  "
          f"F1 micro={row['f1_micro']:.4f} ({row['f1_micro_retention']:.1f}%)")